# 🌲 YOLOv8 Smoke and Forest Fire Detection Model Training
This notebook provides a complete guide to training a custom **YOLOv8** object detection model for smoke and forest fire verification on satellite and aerial imagery. 

### Platform Recommendation
To train this model on a zero-dollar budget:
1. Open **[Kaggle](https://www.kaggle.com/)** or **[Google Colab](https://colab.research.google.com/)**.
2. Upload this `.ipynb` file.
3. Set your runtime Accelerator to **GPU (T4 or P100)** to train in minutes instead of hours.

### Step 1: Install Ultralytics and Roboflow
We install the official `ultralytics` package (which packages YOLOv8) and `roboflow` to fetch public annotated datasets.

In [ ]:
!pip install -q ultralytics roboflow

### Step 2: Verify GPU Availability
Verify that the GPU environment is enabled and active.

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

### Step 3: Fetch Open-Source Wildfire Dataset
To avoid manual annotation, we can download a public dataset like the **"Wildfire Smoke Detection"** or **"FLAME Dataset"** from Roboflow Universe.

Create a free account at [Roboflow](https://roboflow.com/) to get an API key.

In [ ]:
from roboflow import Roboflow

# Replace with your actual Roboflow API key
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")

# Download a public satellite/aerial wildfire smoke dataset formatted for YOLOv8
project = rf.workspace("roboflow-universe-open-datasets").project("wildfire-smoke-detection-lsjpy")
dataset = project.version(1).download("yolov8")

### Step 4: Initialize and Train YOLOv8 Model
We initialize a pre-trained **YOLOv8 Nano (`yolov8n.pt`)** model. The Nano variant is selected because it is highly lightweight, runs extremely fast on CPU-based runners (like GitHub Actions), and matches the low-bandwidth requirements of our zero-budget cron jobs.

In [ ]:
# Load pre-trained weights to start transfer learning
model = YOLO('yolov8n.pt')

# Train the model for 50 epochs (imgsz 640 is default)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,  # Use GPU
    workers=2,
    plots=True
)

### Step 5: Evaluate Model Precision & Recall

In [ ]:
# Validate on the validation dataset split
metrics = model.val()
print(f"Mean Average Precision (mAP) 50-95: {metrics.box.map:.4f}")
print(f"mAP @ 50: {metrics.box.map50:.4f}")

### Step 6: Test Detection on a Custom Image
Inspect how well your model highlights smoke plumes on raw test imagery.

In [ ]:
import glob
import os
from PIL import Image

# Grab a sample image from the test set
test_images = glob.glob(f"{dataset.location}/test/images/*")
if test_images:
    sample_path = test_images[0]
    print(f"Testing inference on: {sample_path}")
    
    # Run detection
    test_results = model(sample_path)
    
    # Save and view the annotated image
    test_results[0].save(filename="test_output.jpg")
    display(Image.open("test_output.jpg"))
else:
    print("No test images found.")

### Step 7: Export Model for Early Warning Deployment
Once training completes, download your custom weights file from the environment:

1. In your Kaggle/Colab file explorer, navigate to: 
   `runs/detect/train/weights/best.pt`
2. Download `best.pt` to your computer.
3. Rename it to **`model.pt`**.
4. Place `model.pt` in the root of your project directory.

The early warning pipeline will automatically detect the presence of `model.pt` and switch from simulation mode to active AI object-detection inference!